<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/neural_importance_sampling_asimov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 6 — Neural importance sampling for efficient unbinned Asimov data

Exercise 5 constructed an unbinned Asimov dataset by drawing points from the
learned reference density $q_\phi(x)$ and assigning them deterministic weights.
For ordinary independent sampling, the numerical uncertainty of an expected
test-statistic scan decreases only as $M^{-1/2}$.

In this exercise we ask a sharper question:

> Can a second generative network learn where the Asimov integrand is large,
> so that a much smaller weighted sample reproduces the same expected scan?

The idea is neural importance sampling. It is related to the strategy used in
[SPINUP](https://arxiv.org/abs/2507.15084), although the integral is different:
SPINUP learns a conditional proposal for a forward-folding integral, whereas
here we learn one unconditional proposal for the complete Asimov likelihood scan.

This notebook **loads** the expensive reference flow, PRESEL classifier, and
signal/reference and background/reference ensembles saved by Exercise 5. It
never retrains them. The only new model is a dedicated normalizing flow used as
an importance proposal. Every figure is also exported as a self-contained
Python script in `exercise6_figures_scripts/`.



In [ ]:
## ============================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/hnsbi-toolkit.git"
BRANCH = "main"
N_BKG, N_SIG = 100_000_000, 20_000_000
USE_DRIVE = True
REMAKE_EVENTS = False


def run(*args, env=None):
    """Run one setup command and stop immediately if it fails."""
    subprocess.run([str(arg) for arg in args], check=True, env=env)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/hsbi-toolkit")
    else:
        ROOT = Path("/content/hsbi-toolkit")
    ROOT.mkdir(parents=True, exist_ok=True)

    # Keep the source checkout separate from the shared paper-run artifacts.
    REPO_DIR = ROOT / "hnsbi-toolkit"
    NOTEBOOK_DIR = REPO_DIR / "examples" / "notebooks"
    WORK_DIR = ROOT / "paper-examples"

    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)

    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "schemas", "examples/notebooks",
    )
    run(
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{REPO_DIR}[data,flows,lhc,plots]",
    )

    for import_dir in (REPO_DIR / "src", NOTEBOOK_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)

    remake_events = globals().get("REMAKE_EVENTS", False)
    if remake_events or not Path("dataframes/signal.parquet").exists():
        run(
            sys.executable,
            NOTEBOOK_DIR / "generate_distributions.py",
            "--n_bkg", N_BKG,
            "--n_sig", N_SIG,
        )
else:
    for candidate in [Path.cwd(), Path.cwd() / "examples" / "notebooks"]:
        if (candidate / "utils_nf.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

print("Working dir:", os.getcwd())


## Statistical target

Exercise 5 writes the hybrid event intensity as

$$
\nu(x\mid\mu)=q_\phi(x)H_\mu(x),\qquad
H_\mu(x)=\mu\lambda_S r_S(x)+\lambda_B r_B(x).
$$

For an Asimov dataset generated at $\mu_A=1$, the expected test statistic is

$$
t_A(\mu)=2(\Lambda_\mu-\Lambda_A)
+2\,\mathbb E_{q_\phi}\!\left[Y_\mu(x)\right],
\qquad
Y_\mu(x)=H_A(x)\log\frac{H_A(x)}{H_\mu(x)}.
$$

If events are instead drawn from a proposal $g(x)$, the same expectation is

$$
\mathbb E_{q_\phi}[Y_\mu]
=\mathbb E_g\!\left[\frac{q_\phi(x)}{g(x)}Y_\mu(x)\right].
$$

The actual finite-sample calculation is self-normalized: the proposal
weights sum to one and both process ratios are normalized on the same
sample. The variance is therefore controlled by the influence function
of this complete plug-in estimator, not by the uncentered $Y_\mu$ alone.
If $a_j(x)$ is a raw learned ratio, $m_j=\mathbb E_{q_\phi}[a_j]$,
$r_j=a_j/m_j$, $F_\mu=H_A\log(H_A/H_\mu)$, and
$I_\mu=\mathbb E_{q_\phi}[F_\mu]$, its influence function is

$$
\psi_\mu(x)=F_\mu(x)-I_\mu
+\sum_{j\in\{S,B\}}\frac{\partial I_\mu}{\partial m_j}
[a_j(x)-m_j].
$$

The last terms propagate the fact that the two ratio normalizations are
estimated from the same finite sample. For a grid of scan points
$\{\mu_k\}$, minimizing the average asymptotic variance gives

$$
g^*(x)\propto q_\phi(x)A(x),\qquad
A(x)=\left[\frac{1}{K}\sum_{k=1}^K \psi_{\mu_k}^2(x)\right]^{1/2}.
$$

We approximate this more aggressive, variance-matched target with a new
normalizing flow $g_\psi$ using weighted maximum likelihood. A defensive
mixture

$$
g_\epsilon(x)=(1-\epsilon)g_\psi(x)+\epsilon q_\phi(x)
$$
guarantees reference-space coverage and bounds the largest importance weight
by $1/\epsilon$.



In [ ]:
import gc
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.special import logsumexp

import torch

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_plotting import export_standalone_figure_script
from utils_nf import (
    accumulate_preselection_histogram,
    checkpoint_path,
    choose_preselection_ratio_cut,
    flow_log_prob_x,
    flow_sample_x,
    load_flow,
    train_flow,
)

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")



## Configuration and Exercise 5 checkpoints

The paths below are exactly those used by Exercise 5. If any checkpoint is
missing, run Exercise 5 through the two density-ratio training sections first.
The checkpoints persist across Colab sessions when both notebooks use Drive.

The default event counts are deliberately much smaller than the five-million
point reference sample used in Exercise 5. Increase the benchmark and number
of repetitions only after the complete exercise runs once.



In [ ]:
BASE_PATH = Path("./dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}

NIS_MODEL_DIR = Path("models_flows_asimov_nis_influence_v2")
NIS_PLOT_DIR = Path("plots_asimov_nis_influence_v2")
NIS_CACHE_DIR = Path("saved_asimov_nis_influence_v2")
FIGURE_SCRIPT_DIR = Path("exercise6_figures_scripts")
for directory in [NIS_MODEL_DIR, NIS_PLOT_DIR, NIS_CACHE_DIR, FIGURE_SCRIPT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}

# Exercise 5 PRESEL definition.
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100_000
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)

# Saved Exercise 5 ensemble and flow architecture.
RATIO_ENSEMBLE_SIZE = 4
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65_536

# Neural importance-sampling design.
ASIMOV_MU_TRUE = 1.0
MU_DESIGN = np.linspace(0.0, 3.0, 13)
MU_SCAN = np.linspace(0.0, 3.0, 61)
PILOT_EVENTS = 2_000_000
NIS_TARGET_CLIP_QUANTILE = 0.9999
NIS_TARGET_FLOOR_FRACTION = 1.0e-5
DEFENSIVE_REFERENCE_FRACTION = 0.02
DEFENSIVE_FRACTION_CANDIDATES = (0.10, 0.05, 0.02, 0.01)
ACCEPTANCE_CALIBRATION_EVENTS = 250_000

# Proof-of-principle study.
BENCHMARK_EVENTS = 1_000_000
BENCHMARK_BLOCKS = 10
STUDY_SAMPLE_SIZES = np.asarray(
    [512, 1_024, 2_048, 4_096, 8_192, 16_384, 32_768], dtype=int
)
N_REPETITIONS = 64
SHOWCASE_SAMPLE_SIZE = 2_048

# This deliberately aggressive proposal has capacity comparable to the
# reference flow. Exact importance weights still correct residual errors.
NIS_MODEL_CONFIG = {
    "flow_type": "quadratic_spline",
    "n_features": N_DIM,
    "n_coupling_layers": 12,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "scale_clip": 1.5,
    "spline_num_bins": 24,
    "spline_tail_bound": 6.0,
    "dropout_probability": 0.0,
}
NIS_TRAINING_CONFIG = {
    "batch_size": 4096,
    "n_epochs": 70,
    "learning_rate": 1.0e-4,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-7,
    "weight_decay": 0.0,
    "validation_fraction": 0.20,
    "patience": 10,
    "gradient_clip": 5.0,
}


def export_exercise6_figure(fig, script_name):
    """Write a self-contained, editable script for one completed figure."""
    return export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )


print(f"Standalone figure scripts will be written to {FIGURE_SCRIPT_DIR}/")

required_paths = [
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
    checkpoint_path("reference", REFERENCE_FLOW_MODEL_DIR, REFERENCE_FLOW_TYPE),
]
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    for member in range(RATIO_ENSEMBLE_SIZE):
        required_paths.extend(
            [model_dir / f"model{member}.onnx", model_dir / f"model_scaler{member}.bin"]
        )

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    formatted = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Exercise 6 only loads Exercise 5 models. Missing checkpoints:\n"
        f"{formatted}\nRun Exercise 5 through both ratio trainings first."
    )
print("All Exercise 5 checkpoints are available.")



## Load the PRESEL classifier and reconstruct its saved selection

Exercise 5 selected the PRESEL threshold from a streamed weighted histogram.
The trained model is saved, but the threshold is not part of its ONNX file.
On the first Exercise 6 run we therefore repeat only this inexpensive,
memory-bounded histogram pass and cache the resulting threshold and yields.
No network is retrained and no large dataframe is retained.



In [ ]:
def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    available = ort.get_available_providers()
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in available
    ] or available
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


PRESEL_scaler, PRESEL_model_proto = load_trained_model(
    PRESEL_MODEL_DIR / "model0.onnx",
    PRESEL_MODEL_DIR / "model_scaler0.bin",
)
PRESEL_model = as_inference_session(PRESEL_model_proto)
del PRESEL_model_proto


def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=np.float64).reshape(-1)


PRESEL_STATE_PATH = NIS_CACHE_DIR / "exercise5_preselection_state.npz"
if PRESEL_STATE_PATH.exists():
    state = np.load(PRESEL_STATE_PATH)
    PRESEL_RATIO_CUT = float(state["ratio_cut"])
    LAM_SIG = float(state["lambda_signal"])
    LAM_BKG = float(state["lambda_background"])
    print(f"Loaded cached PRESEL state from {PRESEL_STATE_PATH}")
else:
    edges = np.linspace(
        PRESEL_LOG_RATIO_RANGE[0],
        PRESEL_LOG_RATIO_RANGE[1],
        PRESEL_CUT_HISTOGRAM_BINS + 1,
    )
    histograms = {}
    statistics = {}
    for sample_name in ["signal", "background"]:
        histograms[sample_name], statistics[sample_name] = (
            accumulate_preselection_histogram(
                SAMPLE_PATHS[sample_name],
                features=FEATURES,
                ratio_predictor=evaluate_PRESEL_ratio,
                log_ratio_edges=edges,
                batch_size=STREAM_BATCH_SIZE,
                presel_fraction=PRESEL_TRAIN_FRACTION,
                flow_train_fraction=FLOW_TRAIN_FRACTION,
                split_seed=SPLIT_SEED,
            )
        )

    PRESEL_RATIO_CUT, diagnostics = choose_preselection_ratio_cut(
        histograms["signal"],
        histograms["background"],
        edges,
        signal_inclusive_yield=statistics["signal"]["inclusive_weight"],
        background_inclusive_yield=statistics["background"]["inclusive_weight"],
        signal_partition_weight=statistics["signal"]["partition_weight"],
        background_partition_weight=statistics["background"]["partition_weight"],
        target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
    )
    LAM_SIG = diagnostics["histogram_signal_yield"]
    LAM_BKG = diagnostics["histogram_background_yield"]
    np.savez(
        PRESEL_STATE_PATH,
        ratio_cut=PRESEL_RATIO_CUT,
        lambda_signal=LAM_SIG,
        lambda_background=LAM_BKG,
    )
    print(f"Saved PRESEL state to {PRESEL_STATE_PATH}")

print(f"PRESEL ratio cut: {PRESEL_RATIO_CUT:.6g}")
print(f"Post-selection yields: signal={LAM_SIG:.6g}, background={LAM_BKG:.6g}")
print(f"Post-selection B/S: {LAM_BKG / LAM_SIG:.3f}")



## Load the hybrid model from Exercise 5

We load the reference flow directly from its PyTorch checkpoint and each
member of the two ONNX density-ratio ensembles. Predictions are averaged as
ratios, exactly as in Exercise 5. Histogram calibration remains disabled.



In [ ]:
reference_flow = load_flow(
    "reference",
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    flow_type=REFERENCE_FLOW_TYPE,
    device=device,
    expected_features=FEATURES,
)

ratio_models = {}
for sample_name, model_dir in RATIO_MODEL_DIR.items():
    ratio_models[sample_name] = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        scaler, model_proto = load_trained_model(
            model_dir / f"model{member}.onnx",
            model_dir / f"model_scaler{member}.bin",
        )
        ratio_models[sample_name].append(
            {"scaler": scaler, "model": as_inference_session(model_proto)}
        )


def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    values = np.asarray(values, dtype=np.float32)
    chunks = []
    for start in range(0, len(values), int(batch_size)):
        batch = pd.DataFrame(values[start : start + int(batch_size)], columns=FEATURES)
        member_predictions = []
        for pack in ratio_models[sample_name]:
            prediction = predict_with_model(
                batch,
                scaler=pack["scaler"],
                model=pack["model"],
            )
            member_predictions.append(
                np.asarray(prediction, dtype=np.float64).reshape(-1)
            )
        chunks.append(np.mean(np.stack(member_predictions, axis=0), axis=0))
    ratio = np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float64)
    if not np.isfinite(ratio).all():
        raise FloatingPointError(f"The {sample_name} ratio returned NaN or infinity.")
    return np.maximum(ratio, RATIO_FLOOR)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted_chunks.append(generated[passes])
            n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


def conditional_log_prob(flow_pack, values, acceptance, batch_size=65_536):
    return (
        np.asarray(
            flow_log_prob_x(flow_pack, values, batch_size=batch_size),
            dtype=np.float64,
        )
        - np.log(float(acceptance))
    )


print(f"Loaded reference flow: {reference_flow['path']}")
print("Loaded four signal/reference and four background/reference models.")



## Build a pilot approximation to the optimal proposal

A pilot sample from $q_\phi$ serves two purposes. First, it fixes the
normalization of the learned ratios. Second, it evaluates the scan-wide
importance amplitude $A(x)$. The overall scale of $A$ is irrelevant.

We clip only the extreme upper tail used for proposal training and add a very
small floor. These regularizations prevent a handful of pilot events from
dominating the weighted likelihood. They cannot bias the final Asimov integral because
the exact factor $q_\phi/g_\epsilon$ is applied later.



In [ ]:
def normalized_ratios(raw_signal, raw_background, weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
    signal_norm = float(np.sum(weights * raw_signal))
    background_norm = float(np.sum(weights * raw_background))
    return raw_signal / signal_norm, raw_background / background_norm


def scan_influence_amplitude(
    raw_signal, raw_background, mu_values, weights=None
):
    """Influence amplitude of the self-normalized, ratio-normalized scan."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()

    mean_signal = float(np.sum(weights * raw_signal))
    mean_background = float(np.sum(weights * raw_background))
    ratio_signal = raw_signal / mean_signal
    ratio_background = raw_background / mean_background
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal
        + LAM_BKG * ratio_background
    )
    amplitude_squared = np.zeros(len(ratio_signal), dtype=np.float64)
    scale = ASIMOV_MU_TRUE * LAM_SIG + LAM_BKG
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        log_h_ratio = np.log(h_asimov / h_mu)
        f_mu = h_asimov * log_h_ratio
        integral_mu = float(np.sum(weights * f_mu))

        # Derivatives of F_mu with respect to the normalized ratios.
        h_ratio = h_asimov / h_mu
        derivative_signal = (
            ASIMOV_MU_TRUE * LAM_SIG * (log_h_ratio + 1.0)
            - mu * LAM_SIG * h_ratio
        )
        derivative_background = LAM_BKG * (
            log_h_ratio + 1.0 - h_ratio
        )
        derivative_mean_signal = float(
            np.sum(
                weights * derivative_signal * (-ratio_signal / mean_signal)
            )
        )
        derivative_mean_background = float(
            np.sum(
                weights
                * derivative_background
                * (-ratio_background / mean_background)
            )
        )
        influence_mu = (
            f_mu
            - integral_mu
            + derivative_mean_signal * (raw_signal - mean_signal)
            + derivative_mean_background
            * (raw_background - mean_background)
        )
        amplitude_squared += (influence_mu / scale) ** 2
    return np.sqrt(amplitude_squared / len(mu_values))


torch.manual_seed(SEED + 100)
pilot_values, REFERENCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    reference_flow,
    PILOT_EVENTS,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
pilot_raw_signal = evaluate_ratio("signal", pilot_values)
pilot_raw_background = evaluate_ratio("background", pilot_values)
RATIO_NORMALIZATION = {
    "signal": float(np.mean(pilot_raw_signal)),
    "background": float(np.mean(pilot_raw_background)),
}

pilot_amplitude = scan_influence_amplitude(
    pilot_raw_signal, pilot_raw_background, MU_DESIGN
)
positive_amplitude = pilot_amplitude[pilot_amplitude > 0.0]
amplitude_floor = (
    NIS_TARGET_FLOOR_FRACTION * float(np.median(positive_amplitude))
)
amplitude_ceiling = float(
    np.quantile(pilot_amplitude, NIS_TARGET_CLIP_QUANTILE)
)
pilot_training_amplitude = np.clip(
    pilot_amplitude, amplitude_floor, amplitude_ceiling
)

print(f"Pilot reference events: {len(pilot_values):,}")
print(f"Reference PRESEL acceptance: {REFERENCE_PRESEL_ACCEPTANCE:.4%}")
print("Ratio normalizations:", RATIO_NORMALIZATION)
print(
    "Importance-amplitude quantiles:",
    np.quantile(pilot_amplitude, [0.0, 0.5, 0.9, 0.99, 0.999, 1.0]),
)
print(
    f"Training amplitude floor/ceiling: "
    f"{amplitude_floor:.3e}/{amplitude_ceiling:.3e}"
)



## Train the neural importance proposal

The pilot events are fitted with weights proportional to $A(x)$. Weighted
maximum likelihood targets $q_\phi(x)A(x)$ directly without producing
duplicate events through resampling. This reduces an avoidable source of
training noise in the tails and uses all two million pilot points.

The proposal checkpoint is independent of the Exercise 5 checkpoints. Once
trained, it is loaded automatically on subsequent runs.



In [ ]:
nis_checkpoint = checkpoint_path(
    "asimov_importance", NIS_MODEL_DIR, NIS_MODEL_CONFIG["flow_type"]
)
if nis_checkpoint.exists():
    importance_flow = load_flow(
        "asimov_importance",
        model_dir=NIS_MODEL_DIR,
        flow_type=NIS_MODEL_CONFIG["flow_type"],
        device=device,
        expected_features=FEATURES,
    )
    print(f"Loaded neural importance proposal from {nis_checkpoint}")
else:
    importance_training_df = pd.DataFrame(
        pilot_values, columns=FEATURES
    )
    importance_flow = train_flow(
        "asimov_importance",
        importance_training_df,
        features=FEATURES,
        model_dir=NIS_MODEL_DIR,
        model_config=NIS_MODEL_CONFIG,
        training_config=NIS_TRAINING_CONFIG,
        device=device,
        sample_weights=pilot_training_amplitude,
        max_train_events=None,
        load_if_available=True,
        seed=SEED + 201,
    )
    del importance_training_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# The proposal has absorbed the pilot information; release the large arrays
# before constructing independent validation and benchmark samples.
del (
    pilot_values,
    pilot_raw_signal,
    pilot_raw_background,
    pilot_amplitude,
    pilot_training_amplitude,
    positive_amplitude,
)
gc.collect()

# Estimate the acceptance needed to evaluate the proposal density conditioned
# on the same PRESEL region as the reference density.
torch.manual_seed(SEED + 300)
acceptance_probe_values, IMPORTANCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    importance_flow,
    ACCEPTANCE_CALIBRATION_EVENTS,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
del acceptance_probe_values
gc.collect()
print(f"Importance-flow PRESEL acceptance: {IMPORTANCE_PRESEL_ACCEPTANCE:.4%}")



## Validation 1: did the proposal learn the desired target?

The ideal relation is

$$
\log\frac{g_\psi(x)}{q_\phi(x)}
=\log A(x)+\mathrm{constant}.
$$

We test this relation on an independent reference sample. Since only the
shape matters, both sides are centered before comparing them. A strong linear
relation with unit slope demonstrates that the proposal has moved probability
toward the regions that dominate the scan.



In [ ]:
VALIDATION_EVENTS = 200_000
torch.manual_seed(SEED + 400)
validation_values, _ = sample_preselected_flow(
    reference_flow, VALIDATION_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
validation_raw_signal = evaluate_ratio("signal", validation_values)
validation_raw_background = evaluate_ratio("background", validation_values)
validation_amplitude = scan_influence_amplitude(
    validation_raw_signal, validation_raw_background, MU_DESIGN
)
validation_amplitude = np.clip(
    validation_amplitude, amplitude_floor, amplitude_ceiling
)

validation_log_q = conditional_log_prob(
    reference_flow, validation_values, REFERENCE_PRESEL_ACCEPTANCE
)
validation_log_g = conditional_log_prob(
    importance_flow, validation_values, IMPORTANCE_PRESEL_ACCEPTANCE
)
log_target = np.log(validation_amplitude)
log_proposal_ratio = validation_log_g - validation_log_q
log_target_centered = log_target - np.mean(log_target)
log_proposal_centered = log_proposal_ratio - np.mean(log_proposal_ratio)
proposal_target_correlation = float(
    np.corrcoef(log_target_centered, log_proposal_centered)[0, 1]
)
proposal_target_slope = float(
    np.polyfit(log_target_centered, log_proposal_centered, deg=1)[0]
)
proposal_target_rmse = float(
    np.sqrt(np.mean((log_proposal_centered - log_target_centered) ** 2))
)

# Predict the scan-wide variance gain for several defensive fractions
# using held-out q_phi points. Since the influence functions are centered,
# E_q[(q/g) A^2] is the relevant average asymptotic variance.
proposal_to_reference = np.exp(np.clip(log_proposal_ratio, -80.0, 80.0))
direct_second_moment = float(np.mean(validation_amplitude**2))
defensive_rows = []
for epsilon in DEFENSIVE_FRACTION_CANDIDATES:
    q_over_mix = 1.0 / (
        epsilon + (1.0 - epsilon) * proposal_to_reference
    )
    proposal_second_moment = float(
        np.mean(q_over_mix * validation_amplitude**2)
    )
    defensive_rows.append(
        {
            "epsilon": epsilon,
            "predicted_variance_gain": direct_second_moment
            / proposal_second_moment,
            "q_over_g_q99.9": float(np.quantile(q_over_mix, 0.999)),
            "q_over_g_max": float(np.max(q_over_mix)),
            "hard_bound": 1.0 / epsilon,
        }
    )
defensive_diagnostics = pd.DataFrame(defensive_rows)

rng = np.random.default_rng(SEED + 401)
plot_indices = rng.choice(
    len(validation_values), size=min(40_000, len(validation_values)), replace=False
)
limit = np.quantile(
    np.abs(
        np.concatenate(
            [log_target_centered[plot_indices], log_proposal_centered[plot_indices]]
        )
    ),
    0.995,
)

fig, ax = plt.subplots(figsize=(6.4, 5.5))
hexbin = ax.hexbin(
    log_target_centered[plot_indices],
    log_proposal_centered[plot_indices],
    gridsize=70,
    bins="log",
    mincnt=1,
    cmap="viridis",
)
ax.plot([-limit, limit], [-limit, limit], color="black", ls="--", lw=1.5)
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_xlabel(r"Centered $\log A(x)$")
ax.set_ylabel(r"Centered $\log[g_\psi(x)/q_\phi(x)]$")
ax.set_title("Neural proposal versus variance-optimal target")
ax.text(
    0.04,
    0.96,
    rf"$\rho={proposal_target_correlation:.3f}$"
    + "\n"
    + rf"slope$={proposal_target_slope:.3f}$"
    + "\n"
    + rf"RMS$={proposal_target_rmse:.3f}$",
    transform=ax.transAxes,
    va="top",
)
fig.colorbar(hexbin, ax=ax, label="log count")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "proposal_target_closure.png", dpi=160)
export_exercise6_figure(fig, "proposal_target_closure")
plt.show()

print(f"Proposal/target correlation: {proposal_target_correlation:.6f}")
print(f"Proposal/target slope: {proposal_target_slope:.6f}")
print(f"Centered log-ratio RMS: {proposal_target_rmse:.6f}")
display(defensive_diagnostics)
selected_row = defensive_diagnostics.loc[
    np.isclose(
        defensive_diagnostics["epsilon"], DEFENSIVE_REFERENCE_FRACTION
    )
].iloc[0]
print(
    f"Using epsilon={DEFENSIVE_REFERENCE_FRACTION:g}; held-out predicted "
    f"variance gain={selected_row['predicted_variance_gain']:.3f}."
)



## Defensive proposal and exact importance weights

We now sample from $g_\epsilon=(1-\epsilon)g_\psi+\epsilon q_\phi$.
Both component densities are evaluated after conditioning on PRESEL, so the
importance weights are known event by event. We use normalized quadrature
weights in finite samples. This introduces the familiar
$\mathcal O(M^{-1})$ self-normalization bias but is numerically stable and
guarantees that the quadrature integrates a constant exactly.



In [ ]:

def log_defensive_proposal(values):
    log_q = conditional_log_prob(
        reference_flow, values, REFERENCE_PRESEL_ACCEPTANCE
    )
    log_g = conditional_log_prob(
        importance_flow, values, IMPORTANCE_PRESEL_ACCEPTANCE
    )
    log_mix = logsumexp(
        np.stack(
            [
                np.log(DEFENSIVE_REFERENCE_FRACTION) + log_q,
                np.log1p(-DEFENSIVE_REFERENCE_FRACTION) + log_g,
            ],
            axis=0,
        ),
        axis=0,
    )
    return log_mix, log_q


def sample_defensive_proposal(n_events, seed):
    rng = np.random.default_rng(seed)
    n_reference = int(rng.binomial(int(n_events), DEFENSIVE_REFERENCE_FRACTION))
    n_importance = int(n_events) - n_reference

    torch.manual_seed(seed + 1)
    reference_values, _ = sample_preselected_flow(
        reference_flow, n_reference, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    ) if n_reference else (np.empty((0, N_DIM), dtype=np.float32), np.nan)
    torch.manual_seed(seed + 2)
    importance_values, _ = sample_preselected_flow(
        importance_flow, n_importance, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    ) if n_importance else (np.empty((0, N_DIM), dtype=np.float32), np.nan)

    values = np.concatenate([reference_values, importance_values], axis=0)
    values = values[rng.permutation(len(values))]
    log_mix, log_q = log_defensive_proposal(values)
    return values, log_q - log_mix


def normalized_importance_weights(log_weights):
    log_weights = np.asarray(log_weights, dtype=np.float64)
    return np.exp(log_weights - logsumexp(log_weights))


torch.manual_seed(SEED + 500)
proposal_validation_values, proposal_validation_log_weights = (
    sample_defensive_proposal(VALIDATION_EVENTS, SEED + 500)
)
proposal_validation_weights = normalized_importance_weights(
    proposal_validation_log_weights
)
proposal_validation_ess = 1.0 / np.sum(proposal_validation_weights**2)

fig, axes = plt.subplots(1, N_DIM, figsize=(3.2 * N_DIM, 3.2))
for axis, feature_index, feature_name in zip(axes, range(N_DIM), FEATURES):
    combined = np.concatenate(
        [validation_values[:, feature_index], proposal_validation_values[:, feature_index]]
    )
    edges = np.quantile(combined, np.linspace(0.001, 0.999, 41))
    edges = np.unique(edges)
    axis.hist(
        validation_values[:, feature_index],
        bins=edges,
        density=True,
        histtype="step",
        lw=2,
        label=r"Direct $q_\phi$",
    )
    axis.hist(
        proposal_validation_values[:, feature_index],
        bins=edges,
        weights=proposal_validation_weights,
        density=True,
        histtype="step",
        lw=1.8,
        label=r"$g_\epsilon$, weighted",
    )
    axis.set_xlabel(feature_name)
    axis.set_ylabel("Density")
axes[0].legend(fontsize=8)
fig.suptitle("Importance-reweighted reference closure", y=1.02)
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "importance_reweighted_reference_closure.png", dpi=160)
export_exercise6_figure(fig, "importance_reweighted_reference_closure")
plt.show()

print(
    f"Defensive-proposal ESS: {proposal_validation_ess:,.0f}/"
    f"{VALIDATION_EVENTS:,} ({proposal_validation_ess / VALIDATION_EVENTS:.1%})"
)
print(
    "Raw q/g weight quantiles:",
    np.quantile(
        np.exp(proposal_validation_log_weights),
        [0.0, 0.5, 0.9, 0.99, 0.999, 1.0],
    ),
)

del (
    validation_values,
    validation_raw_signal,
    validation_raw_background,
    validation_amplitude,
    validation_log_q,
    validation_log_g,
    proposal_validation_values,
    proposal_validation_log_weights,
    proposal_validation_weights,
)
gc.collect()



## Weighted Asimov scan on a general quadrature

Every finite quadrature uses weights $\omega_m$ normalized to one. Before
constructing the scan, each learned process ratio is normalized on that same
quadrature,

$$
\widetilde r_j(x_m)=
\frac{\widehat r_j(x_m)}{\sum_n\omega_n\widehat r_j(x_n)}.
$$

Consequently, the Asimov score at $\mu_A=1$ vanishes exactly for both direct
and importance sampling. The comparison below therefore probes the difficult
part of the calculation—the curvature and finite displacement of the scan—
rather than a trivial shift of its minimum.



In [ ]:
def asimov_scan(raw_signal, raw_background, mu_values, log_weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)

    ratio_signal, ratio_background = normalized_ratios(
        raw_signal, raw_background, weights
    )
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal
        + LAM_BKG * ratio_background
    )

    scan = []
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        statistic = 2.0 * (
            (mu - ASIMOV_MU_TRUE) * LAM_SIG + integral
        )
        scan.append(max(0.0, float(statistic)))

    score_at_truth = 2.0 * LAM_SIG * (
        1.0 - float(np.sum(weights * ratio_signal))
    )
    ess = 1.0 / np.sum(weights**2)
    return np.asarray(scan), float(score_at_truth), float(ess)


def evaluate_hybrid_ratios(values):
    return evaluate_ratio("signal", values), evaluate_ratio("background", values)


print("Constructing a high-statistics numerical benchmark...")
torch.manual_seed(SEED + 600)
benchmark_values, _ = sample_preselected_flow(
    reference_flow, BENCHMARK_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
benchmark_raw_signal, benchmark_raw_background = evaluate_hybrid_ratios(
    benchmark_values
)
del benchmark_values
gc.collect()
BENCHMARK_SCAN, BENCHMARK_SCORE, BENCHMARK_ESS = asimov_scan(
    benchmark_raw_signal, benchmark_raw_background, MU_SCAN
)
zero_index = int(np.argmin(np.abs(MU_SCAN)))
BENCHMARK_Q_ZERO = float(BENCHMARK_SCAN[zero_index])
BENCHMARK_SIGMA = ASIMOV_MU_TRUE / np.sqrt(BENCHMARK_Q_ZERO)

# A block estimate makes clear that this is a precise numerical benchmark, not
# an analytic truth curve.
benchmark_block_q0 = []
for block in np.array_split(np.arange(BENCHMARK_EVENTS), BENCHMARK_BLOCKS):
    block_scan, _, _ = asimov_scan(
        benchmark_raw_signal[block], benchmark_raw_background[block], MU_SCAN
    )
    benchmark_block_q0.append(block_scan[zero_index])
BENCHMARK_Q_ZERO_SE = float(
    np.std(benchmark_block_q0, ddof=1) / np.sqrt(BENCHMARK_BLOCKS)
)

print(f"Benchmark q_0,A: {BENCHMARK_Q_ZERO:.8f} +/- {BENCHMARK_Q_ZERO_SE:.8f} (blocks)")
print(f"Benchmark sigma_mu: {BENCHMARK_SIGMA:.8f}")
print(f"Benchmark score at mu_A: {BENCHMARK_SCORE:+.3e}")

fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.plot(MU_SCAN, BENCHMARK_SCAN, color="black", lw=2.4)
ax.axvline(ASIMOV_MU_TRUE, color="0.5", ls="--", lw=1.2)
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$t_A(\mu)$")
ax.set_title("High-statistics hybrid-model Asimov benchmark")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "asimov_benchmark_scan.png", dpi=160)
export_exercise6_figure(fig, "asimov_benchmark_scan")
plt.show()



## Proof of principle: equal-cost repeated quadratures

For each repetition we generate the same number of points using either

1. ordinary independent sampling from $q_\phi$, or
2. neural importance sampling from $g_\epsilon$.

The ratios are normalized independently on every finite quadrature. For each
method and sample size we record

* the error on $q_{0,A}$;
* the RMS error of the complete $t_A(\mu)$ scan;
* the inferred $\sigma_\mu=1/\sqrt{q_{0,A}}$;
* the quadrature effective sample size; and
* the score at $\mu_A$, which should vanish to numerical precision.

Neural importance sampling is successful if it remains consistent with the
benchmark while reducing the repeated-sample variance and scan RMSE at fixed
$M$. The variance ratio can be interpreted as an approximate event-saving
factor.



In [ ]:
maximum_study_size = int(np.max(STUDY_SAMPLE_SIZES))
rows = []
showcase_scans = {"Direct reference": [], "Neural importance": []}

for repetition in range(N_REPETITIONS):
    if (repetition + 1) % max(1, N_REPETITIONS // 8) == 0:
        print(f"Completed {repetition + 1}/{N_REPETITIONS} repetitions")

    torch.manual_seed(SEED + 10_000 + repetition)
    direct_values, _ = sample_preselected_flow(
        reference_flow,
        maximum_study_size,
        batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
    )
    direct_raw_signal, direct_raw_background = evaluate_hybrid_ratios(direct_values)

    importance_values, importance_log_weights = sample_defensive_proposal(
        maximum_study_size, SEED + 20_000 + repetition
    )
    importance_raw_signal, importance_raw_background = evaluate_hybrid_ratios(
        importance_values
    )

    for sample_size in STUDY_SAMPLE_SIZES:
        sample_size = int(sample_size)
        for method, raw_signal, raw_background, log_weights in [
            (
                "Direct reference",
                direct_raw_signal[:sample_size],
                direct_raw_background[:sample_size],
                None,
            ),
            (
                "Neural importance",
                importance_raw_signal[:sample_size],
                importance_raw_background[:sample_size],
                importance_log_weights[:sample_size],
            ),
        ]:
            scan, score, ess = asimov_scan(
                raw_signal, raw_background, MU_SCAN, log_weights=log_weights
            )
            q_zero = float(scan[zero_index])
            sigma_mu = ASIMOV_MU_TRUE / np.sqrt(q_zero)
            scan_rmse = float(np.sqrt(np.mean((scan - BENCHMARK_SCAN) ** 2)))
            rows.append(
                {
                    "method": method,
                    "repetition": repetition,
                    "sample_size": sample_size,
                    "q_zero": q_zero,
                    "q_zero_error": q_zero - BENCHMARK_Q_ZERO,
                    "sigma_mu": sigma_mu,
                    "scan_rmse": scan_rmse,
                    "score_at_truth": score,
                    "ess": ess,
                }
            )
            if sample_size == SHOWCASE_SAMPLE_SIZE and repetition < 8:
                showcase_scans[method].append(scan)

study_results = pd.DataFrame(rows)


def summarize_group(group):
    return pd.Series(
        {
            "q0_mean": group["q_zero"].mean(),
            "q0_bias": group["q_zero_error"].mean(),
            "q0_std": group["q_zero"].std(ddof=1),
            "q0_rmse": np.sqrt(np.mean(group["q_zero_error"] ** 2)),
            "scan_rmse": np.sqrt(np.mean(group["scan_rmse"] ** 2)),
            "sigma_rmse": np.sqrt(
                np.mean((group["sigma_mu"] - BENCHMARK_SIGMA) ** 2)
            ),
            "mean_ess": group["ess"].mean(),
            "max_abs_score": np.max(np.abs(group["score_at_truth"])),
        }
    )


summary_rows = []
for (method, sample_size), group in study_results.groupby(
    ["method", "sample_size"], sort=False
):
    row = summarize_group(group).to_dict()
    row.update({"method": method, "sample_size": int(sample_size)})
    summary_rows.append(row)
study_summary = pd.DataFrame(summary_rows)

direct_variance = (
    study_summary.loc[study_summary["method"] == "Direct reference"]
    .set_index("sample_size")["q0_std"] ** 2
)
importance_rows = study_summary[study_summary["method"] == "Neural importance"].copy()
importance_rows["variance_reduction"] = importance_rows["sample_size"].map(
    direct_variance
) / importance_rows["q0_std"] ** 2
study_summary = study_summary.merge(
    importance_rows[["sample_size", "variance_reduction"]],
    on="sample_size",
    how="left",
)

display(
    study_summary[
        [
            "method",
            "sample_size",
            "q0_bias",
            "q0_std",
            "q0_rmse",
            "scan_rmse",
            "mean_ess",
            "max_abs_score",
            "variance_reduction",
        ]
    ]
)



In [ ]:
colors = {"Direct reference": "C3", "Neural importance": "C0"}
markers = {"Direct reference": "o", "Neural importance": "s"}

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for method, group in study_summary.groupby("method", sort=False):
    group = group.sort_values("sample_size")
    axes[0].plot(
        group["sample_size"],
        group["q0_rmse"],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )
    axes[1].plot(
        group["sample_size"],
        group["scan_rmse"],
        marker=markers[method],
        color=colors[method],
        lw=2,
        label=method,
    )

nis_summary = study_summary[
    study_summary["method"] == "Neural importance"
].sort_values("sample_size")
axes[2].plot(
    nis_summary["sample_size"],
    nis_summary["variance_reduction"],
    marker="D",
    color="C2",
    lw=2,
)
axes[2].axhline(1.0, color="black", ls="--", lw=1.2)

for axis in axes[:2]:
    axis.set_xscale("log", base=2)
    axis.set_yscale("log")
    axis.grid(alpha=0.25)
    axis.legend()
axes[2].set_xscale("log", base=2)
axes[2].grid(alpha=0.25)
axes[0].set_xlabel("Number of Asimov points")
axes[0].set_ylabel(r"RMSE of $q_{0,A}$")
axes[0].set_title("Discovery statistic")
axes[1].set_xlabel("Number of Asimov points")
axes[1].set_ylabel(r"RMS error over $t_A(\mu)$")
axes[1].set_title("Complete likelihood scan")
axes[2].set_xlabel("Number of Asimov points")
axes[2].set_ylabel(r"$\mathrm{Var}_{q_\phi}/\mathrm{Var}_{\rm NIS}$")
axes[2].set_title("Approximate event-saving factor")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_asimov_convergence.png", dpi=160)
export_exercise6_figure(fig, "nis_asimov_convergence")
plt.show()



In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.0))
for method in ["Direct reference", "Neural importance"]:
    for index, scan in enumerate(showcase_scans[method]):
        ax.plot(
            MU_SCAN,
            scan,
            color=colors[method],
            alpha=0.20,
            lw=1.2,
            label=method if index == 0 else None,
        )
ax.plot(MU_SCAN, BENCHMARK_SCAN, color="black", lw=2.6, label="Benchmark")
ax.axvline(ASIMOV_MU_TRUE, color="0.5", ls="--", lw=1.0)
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$t_A(\mu)$")
ax.set_title(f"Repeated {SHOWCASE_SAMPLE_SIZE:,}-point Asimov scans")
ax.legend()
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_repeated_small_asimov_scans.png", dpi=160)
export_exercise6_figure(fig, "nis_repeated_small_asimov_scans")
plt.show()



In [ ]:
selected_results = study_results[
    study_results["sample_size"] == SHOWCASE_SAMPLE_SIZE
]
fig, ax = plt.subplots(figsize=(6.5, 4.8))
data = [
    selected_results.loc[selected_results["method"] == method, "q_zero"].to_numpy()
    for method in ["Direct reference", "Neural importance"]
]
box = ax.boxplot(
    data,
    labels=["Direct reference", "Neural importance"],
    patch_artist=True,
    showmeans=True,
)
for patch, color in zip(box["boxes"], [colors["Direct reference"], colors["Neural importance"]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)
ax.axhline(BENCHMARK_Q_ZERO, color="black", lw=2, label="Benchmark")
ax.set_ylabel(r"$q_{0,A}$")
ax.set_title(f"Equal-cost comparison with {SHOWCASE_SAMPLE_SIZE:,} points")
ax.legend()
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "nis_q0_equal_cost.png", dpi=160)
export_exercise6_figure(fig, "nis_q0_equal_cost")
plt.show()



## Interpreting the result

A convincing proof of principle has several simultaneous features:

1. **Proposal closure:** $\log(g_\psi/q_\phi)$ follows $\log A$ on held-out
   reference events.
2. **Importance closure:** proposal events reweighted by $q_\phi/g_\epsilon$
   reproduce the reference feature distributions.
3. **No visible bias:** the mean neural-importance estimate remains compatible
   with the high-statistics benchmark as $M$ increases.
4. **Reduced numerical variance:** the $q_{0,A}$ and complete-scan RMSE are
   smaller than direct reference sampling at equal $M$.
5. **Exact Asimov minimum:** the score at $\mu_A=1$ remains zero up to floating
   point precision because the ratios are normalized on each weighted
   quadrature.

The variance-reduction panel translates the gain into an intuitive number. A
value of ten means that, for estimating $q_{0,A}$, approximately ten times as
many ordinary reference points would be required to match the variance of the
neural-importance construction.

This comparison is conditional on the trained Exercise 5 hybrid model. Drawing
more proposal events reduces quadrature uncertainty, but it does not reduce
modeling uncertainty in the reference flow or density-ratio ensembles.



In [ ]:
showcase_summary = study_summary[
    study_summary["sample_size"] == SHOWCASE_SAMPLE_SIZE
].set_index("method")
showcase_gain = float(
    showcase_summary.loc["Neural importance", "variance_reduction"]
)
showcase_direct_rmse = float(showcase_summary.loc["Direct reference", "q0_rmse"])
showcase_nis_rmse = float(showcase_summary.loc["Neural importance", "q0_rmse"])
maximum_score = float(study_summary["max_abs_score"].max())

print(f"At M={SHOWCASE_SAMPLE_SIZE:,}:")
print(f"  direct q0 RMSE = {showcase_direct_rmse:.6g}")
print(f"  NIS q0 RMSE    = {showcase_nis_rmse:.6g}")
print(f"  variance-reduction factor = {showcase_gain:.3f}")
print(f"Maximum |Asimov score at mu_A| over study = {maximum_score:.3e}")

if showcase_gain > 1.0:
    print(
        "Proof-of-principle result: neural importance sampling reduces the "
        "equal-cost variance for q_0,A."
    )
else:
    print(
        "The proposal is not yet more efficient at the showcase size. Inspect "
        "proposal closure and importance-weight tails before increasing its capacity."
    )



## Suggested exercises

1. Change `MU_DESIGN` so that the proposal targets only $\mu=0$. Compare the
   gain for $q_{0,A}$ with the accuracy of the rest of the scan.
2. Vary `DEFENSIVE_REFERENCE_FRACTION`. Smaller values can improve efficiency,
   but may create unstable weights if the proposal misses a tail.
3. Train the proposal on $|Y_0|$ and verify explicitly that
   $g^*(x)\propto q_\phi(x)|Y_0(x)|$ is optimal for the single $q_{0,A}$
   integral but not necessarily for a complete confidence-interval scan.
4. Replace random latent samples by scrambled Sobol points. Neural importance
   sampling and randomized quasi-Monte Carlo address different sources of
   inefficiency and can be combined.
5. Repeat the proposal training with different random seeds. The final
   importance estimate should remain stable even if proposal quality changes;
   only its variance should change.
